
# Kenya NCD Risk Prediction Model
This notebook simulates health data inspired by Kenyan demographics, trains a machine learning model to predict risk levels for non-communicable diseases (NCDs), and visualizes the results. Ethical considerations are also discussed.


In [ ]:

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")


In [ ]:

# Step 1: Simulate Kenya-inspired health dataset
np.random.seed(42)
n_samples = 500

data = pd.DataFrame({
    'Age': np.random.randint(18, 80, n_samples),
    'BMI': np.random.uniform(18, 35, n_samples),
    'Blood_Pressure': np.random.randint(90, 180, n_samples),
    'Glucose_Level': np.random.randint(70, 200, n_samples),
    'Smoker': np.random.choice([0, 1], n_samples),
    'Alcohol_Consumption': np.random.choice([0, 1], n_samples),
    'Physical_Activity': np.random.randint(0, 5, n_samples),
    'Family_History': np.random.choice([0, 1], n_samples)
})

def assign_risk(row):
    score = 0
    score += (row['BMI'] > 30) + (row['Blood_Pressure'] > 140)
    score += (row['Glucose_Level'] > 150) + row['Smoker'] + row['Alcohol_Consumption']
    score += (row['Physical_Activity'] < 2) + row['Family_History']
    if score <= 2:
        return 0  # Low risk
    elif score <= 4:
        return 1  # Medium risk
    else:
        return 2  # High risk

data['Risk_Level'] = data.apply(assign_risk, axis=1)


In [ ]:

# Step 2: Preprocess the data
X = data.drop("Risk_Level", axis=1)
y = data["Risk_Level"]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:

# Step 3: Train the Random Forest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


In [ ]:

# Step 4: Evaluate the model
y_pred = model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, y_pred))
print("
Classification Report:
", classification_report(y_test, y_pred, zero_division=0))


In [ ]:

# Step 5: Visualize feature importance
importances = model.feature_importances_
features = X.columns
fig1 = px.bar(x=importances, y=features, orientation='h',
              labels={'x': 'Importance', 'y': 'Feature'},
              title='Feature Importance in NCD Risk Prediction')
fig1.show()


In [ ]:

# Step 6: Visualize risk level distribution
fig2 = px.histogram(data, x="Risk_Level", nbins=3,
                    title="Distribution of Risk Levels",
                    labels={"Risk_Level": "Risk Level"})
fig2.show()



## ⚖️ Ethical Considerations

### Bias
- Ensure diverse and representative data to avoid skewed predictions.
- Monitor model performance across different demographic groups.

### Fairness
- Provide transparent explanations for predictions.
- Avoid discrimination by auditing fairness metrics.

### Sustainability
- Use efficient models suitable for low-resource environments.
- Empower local communities with accessible health insights.
